In [ ]:
import os
import numpy as np
import librosa
import json
import time

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.optimizers import Adam

# ===== CONFIG =====
DATASET_PATH = r"C:\Users\ADMIN\Desktop\Sukriti\VIT\SL\AnuAssistant\dataset"
SAMPLE_RATE = 22050
N_MFCC = 20
MAX_PAD_LEN = 40

X_FILE = "X.npy"
Y_FILE = "y.npy"
LABELS_FILE = "labels_2.json"

# ===== LOAD OR EXTRACT FEATURES =====
if os.path.exists(X_FILE) and os.path.exists(Y_FILE):
    print("⚡ Loading cached features...")
    X = np.load(X_FILE)
    y = np.load(Y_FILE)

    with open(LABELS_FILE, "r") as f:
        labels = json.load(f)

else:
    print("📂 Extracting features (first time slow)...")

    X = []
    y = []
    labels = {}
    label_index = 0

    start_time = time.time()

    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if os.path.isdir(folder_path):
            labels[folder] = label_index
            print(f"\n🔹 Processing label {label_index}: {folder}")

            count = 0

            for file in os.listdir(folder_path):
                if file.lower().endswith(".wav"):
                    file_path = os.path.join(folder_path, file)

                    try:
                        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

                        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)

                        if mfcc.shape[1] < MAX_PAD_LEN:
                            pad_width = MAX_PAD_LEN - mfcc.shape[1]
                            mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
                        else:
                            mfcc = mfcc[:, :MAX_PAD_LEN]

                        X.append(mfcc)
                        y.append(label_index)

                        count += 1
                        if count % 50 == 0:
                            print(f"   Processed {count} files...")

                    except Exception as e:
                        print(f"❌ Error: {file_path} → {e}")

            label_index += 1

    X = np.array(X)
    y = to_categorical(y)

    print(f"\n⏱️ Extraction time: {time.time() - start_time:.2f}s")
    print(f"✅ Total samples: {len(X)}")

    # Save cache
    np.save(X_FILE, X)
    np.save(Y_FILE, y)

    with open(LABELS_FILE, "w") as f:
        json.dump(labels, f)

    print("💾 Features cached!")

# ===== NORMALIZE =====
X = (X - np.mean(X)) / (np.std(X) + 1e-6)

# ===== RESHAPE FOR CNN =====
X = X.reshape(X.shape[0], N_MFCC, MAX_PAD_LEN, 1)

# ===== SPLIT =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Data ready")

# ===== MODEL (CNN) =====
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(N_MFCC, MAX_PAD_LEN, 1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(y.shape[1], activation='softmax'))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ===== TRAIN =====
print("\n🚀 Training...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ===== EVALUATE =====
loss, acc = model.evaluate(X_test, y_test)
print(f"\n🎯 Test Accuracy: {acc*100:.2f}%")

# ===== SAVE MODEL (OVERWRITE) =====
model.save("model_2.h5")
print("💾 Model overwritten and saved as model_2.h5")

# ===== SAVE LABELS =====
with open("labels_2.json", "w") as f:
    json.dump(labels, f)

print("📌 Labels saved")